# Gradient Descent

## Learning Objectives
1. Understand how gradient descent optimizes model weights
2. Implement GD variants (batch, SGD, mini-batch)
3. Analyze learning rate and convergence
4. Compare optimizers (momentum, Adam)

## Level 1: Manual Gradient Descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gradient_descent(X, y, learning_rate=0.01, epochs=100):
    '''Basic gradient descent for linear regression.'''
    m = len(y)
    theta = np.zeros(X.shape[1])
    costs = []

    for epoch in range(epochs):
        # Forward
        predictions = X @ theta
        errors = predictions - y

        # Backward
        gradient = (2/m) * X.T @ errors

        # Update
        theta -= learning_rate * gradient

        # Track cost
        cost = np.mean(errors ** 2)
        costs.append(cost)

    return theta, costs

# Generate data
np.random.seed(42)
X = np.c_[np.ones(100), np.random.randn(100, 2)]
true_theta = np.array([1, 2, 3])
y = X @ true_theta + np.random.randn(100) * 0.1

# Train
theta, costs = gradient_descent(X, y, learning_rate=0.1, epochs=50)
print(f"Learned theta: {theta}")
print(f"Final cost: {costs[-1]:.4f}")

# Plot convergence
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(costs)
plt.xlabel('Epoch'), plt.ylabel('MSE Loss')
plt.title('Convergence')

plt.subplot(1, 2, 2)
plt.scatter(y, X @ theta, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('True'), plt.ylabel('Predicted')
plt.title('Predictions')
plt.tight_layout()
plt.show()

## Level 2: Mini-Batch with Momentum

In [ ]:
class GradientDescentOptimizer:
    def __init__(self, learning_rate=0.01, momentum=0.0, batch_size=32):
        self.lr = learning_rate
        self.momentum = momentum
        self.batch_size = batch_size
        self.velocity = None

    def __call__(self, X, y, epochs=100):
        m = len(y)
        theta = np.zeros(X.shape[1])
        self.velocity = np.zeros_like(theta)
        costs = []

        for epoch in range(epochs):
            # Shuffle
            indices = np.random.permutation(m)
            X_shuffled, y_shuffled = X[indices], y[indices]

            epoch_cost = 0
            for i in range(0, m, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]

                # Gradient
                pred = X_batch @ theta
                grad = (2/len(y_batch)) * X_batch.T @ (pred - y_batch)

                # Momentum
                self.velocity = self.momentum * self.velocity - self.lr * grad
                theta += self.velocity

                epoch_cost += np.mean((pred - y_batch) ** 2)

            costs.append(epoch_cost / (m / self.batch_size))

        return theta, costs

# Compare optimizers
X = np.c_[np.ones(200), np.random.randn(200, 5)]
y = np.sum(X[:, 1:3], axis=1) + np.random.randn(200) * 0.1

results = {}
for momentum in [0.0, 0.9]:
    opt = GradientDescentOptimizer(lr=0.1, momentum=momentum, batch_size=32)
    theta, costs = opt(X, y, epochs=50)
    results[f"momentum={momentum}"] = costs

plt.figure(figsize=(10, 5))
for label, costs in results.items():
    plt.plot(costs, label=label)
plt.xlabel('Epoch'), plt.ylabel('Loss')
plt.legend()
plt.title('Impact of Momentum on Convergence')
plt.show()

## Real-World Example 1: Learning Rate Schedules

In [ ]:
def cosine_annealing(epoch, initial_lr=0.1, T_max=100):
    '''Cosine annealing schedule.'''
    return initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / T_max))

def step_decay(epoch, initial_lr=0.1, drop=0.5, epochs_per_drop=10):
    '''Step decay schedule.'''
    return initial_lr * (drop ** (epoch // epochs_per_drop))

# Compare schedules
epochs = 100
schedules = {
    'constant': [0.1] * epochs,
    'cosine': [cosine_annealing(e) for e in range(epochs)],
    'step': [step_decay(e) for e in range(epochs)],
}

plt.figure(figsize=(12, 4))
for name, schedule in schedules.items():
    plt.plot(schedule, label=name)
plt.xlabel('Epoch'), plt.ylabel('Learning Rate')
plt.legend()
plt.title('Learning Rate Schedules')
plt.show()

## Real-World Example 2: Adam Optimizer

In [ ]:
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1, self.beta2 = beta1, beta2
        self.epsilon = epsilon
        self.m, self.v, self.t = None, None, 0

    def __call__(self, theta, gradient):
        if self.m is None:
            self.m = np.zeros_like(theta)
            self.v = np.zeros_like(theta)

        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * gradient
        self.v = self.beta2 * self.v + (1 - self.beta2) * (gradient ** 2)

        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)

        return theta - self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

# Test on non-convex problem
np.random.seed(42)
X = np.c_[np.ones(300), np.random.randn(300, 10)]
y = np.sin(X[:, 1]) + X[:, 2] * X[:, 3] + np.random.randn(300) * 0.1

adam = Adam(lr=0.01)
theta = np.random.randn(X.shape[1]) * 0.1
costs = []

for _ in range(100):
    pred = X @ theta
    grad = (2/len(y)) * X.T @ (pred - y)
    theta = adam(theta, grad)
    costs.append(np.mean((pred - y) ** 2))

plt.plot(costs)
plt.xlabel('Iteration'), plt.ylabel('Loss')
plt.title('Adam Optimizer Convergence')
plt.show()

## Real-World Example 3: Learning Rate Tuning

In [ ]:
def train_with_lr(X, y, learning_rate, epochs=50):
    '''Train and return final loss.'''
    theta = np.zeros(X.shape[1])
    for _ in range(epochs):
        pred = X @ theta
        grad = (2/len(y)) * X.T @ (pred - y)
        theta -= learning_rate * grad
    return np.mean((X @ theta - y) ** 2)

X = np.c_[np.ones(200), np.random.randn(200, 5)]
y = np.sum(X[:, 1:3], axis=1) + np.random.randn(200) * 0.1

# Grid search learning rates
learning_rates = np.logspace(-4, 0, 20)
final_losses = [train_with_lr(X, y, lr) for lr in learning_rates]

plt.figure(figsize=(10, 5))
plt.semilogx(learning_rates, final_losses, 'o-')
plt.xlabel('Learning Rate')
plt.ylabel('Final Loss')
plt.title('Learning Rate vs Final Loss')
plt.axvline(learning_rates[np.argmin(final_losses)], color='r', linestyle='--',
            label=f'Optimal: {learning_rates[np.argmin(final_losses)]:.4f}')
plt.legend()
plt.show()

## Key Takeaways

**When to Use:**
- Gradient descent: foundation for all neural network training
- Batch GD: small datasets, stable but slow
- SGD: large datasets, noisy but fast
- Mini-batch: balanced, most common in practice
- Adam: adaptive learning rates, robust default choice

**Related Concepts:**
- [Backpropagation](./02-backpropagation.md) — computes gradients
- [Optimization Algorithms](./04-optimization-algorithms.md) — advanced variants
- [Learning Rate Scheduling](./05-learning-rate-scheduling.md) — improving convergence